# Thai Sentiment — TF-IDF Training with MLflow

This notebook trains and evaluates TF-IDF based classifiers (SVM, Logistic Regression, XGBoost) on the preprocessed Wisesight sentiment splits, logging parameters, metrics, and artifacts to MLflow.

## 1. Setup & Environment
Install dependencies, mount Google Drive, and set the working directory.

In [ ]:
!pip install mlflow pythainlp boto3

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports

In [4]:
import os
import json
import pickle
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from pythainlp.tokenize import word_tokenize
from dotenv import load_dotenv
from xgboost import XGBClassifier

## 3. Logging Configuration

In [5]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 4. Configuration: Label Names

In [6]:
LABEL_NAMES = ["neg", "neu", "pos", "q"]

## 5. Helper Functions

### 5.1 Thai Tokenizer
Wraps PyThaiNLP's `newmm` tokenizer for use inside the TF-IDF vectorizer.

In [7]:
def thai_tokenize(text: str) -> str:
    tokens = word_tokenize(str(text), engine="newmm", keep_whitespace=False)
    return " ".join(tokens)

### 5.2 Confusion Matrix Plotter

In [8]:
def plot_confusion_matrix(y_true, y_pred, title: str) -> plt.Figure:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    plt.tight_layout()
    return fig

### 5.3 Training & Evaluation Pipeline
Builds a TF-IDF + classifier pipeline (SVM, Logistic Regression, or XGBoost), trains it, evaluates on validation/test sets, and logs everything (params, metrics, confusion matrix, feature importance, classification report, model) to MLflow.

In [9]:
def train_and_evaluate(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_type: str = "svm",
    ngram_range: tuple = (1, 2),
    max_features: int = 100_000,
    C: float = 1.0,
    # ── XGBoost-only params ──────────────────────────────────────
    n_estimators: int = 300,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 0.8,
):
    load_dotenv('.env')
    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
    mlflow.set_experiment(f"thai-sentiment / TF-IDF + {model_type.upper()}")

    # ── Tokenize ──
    logger.info("Tokenizing with PyThaiNLP newmm...")
    train_texts = train_df["text_clean"].apply(thai_tokenize).tolist()
    val_texts   = val_df["text_clean"].apply(thai_tokenize).tolist()
    test_texts  = test_df["text_clean"].apply(thai_tokenize).tolist()

    # ── Build classifier ─────────────────────────────────────────────────────
    if model_type == "svm":
        clf = LinearSVC(C=C, max_iter=2000, random_state=42)

    elif model_type == "logreg":
        clf = LogisticRegression(
            C=C, max_iter=1000, solver="saga", n_jobs=-1, random_state=42
        )

    elif model_type == "xgboost":                       # ← NEW BLOCK
        clf = XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            use_label_encoder=False,
            eval_metric="mlogloss",
            objective="multi:softmax",
            num_class=4,
            n_jobs=-1,
            random_state=42,
            verbosity=0,
        )

    else:
        raise ValueError("model_type must be 'svm', 'logreg', or 'xgboost'")

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=ngram_range,
            min_df=2,
            max_features=max_features,
            sublinear_tf=True,
        )),
        ("clf", clf),
    ])

    with mlflow.start_run(run_name=f"tfidf_{model_type}"):

        # ── Log parameters ───────────────────────────────────────────────────
        params = {
            "model_type":    model_type,
            "tokenizer":     "pythainlp_newmm",
            "ngram_range":   str(ngram_range),
            "max_features":  max_features,
            "train_samples": len(train_df),
            "val_samples":   len(val_df),
            "test_samples":  len(test_df),
        }

        # log SVM/LogReg param only when relevant
        if model_type in ("svm", "logreg"):
            params["C"] = C

        # log XGBoost params only when relevant           ← NEW
        if model_type == "xgboost":
            params.update({
                "n_estimators":  n_estimators,
                "max_depth":     max_depth,
                "learning_rate": learning_rate,
                "subsample":     subsample,
            })

        mlflow.log_params(params)
        mlflow.set_tags({
            "method":   f"TF-IDF + {model_type.upper()}",
            "language": "thai",
            "dataset":  "wisesight_sentiment",
        })

        # ── Train ────────────────────────────────────────────────────────────
        logger.info(f"Training {model_type.upper()}...")

        # XGBoost supports eval_set for early stopping during fit
        if model_type == "xgboost":                       # ← NEW BLOCK
            tfidf    = pipeline.named_steps["tfidf"]
            train_X  = tfidf.fit_transform(train_texts)
            val_X    = tfidf.transform(val_texts)

            pipeline.named_steps["clf"].fit(
                train_X, train_df["label"].tolist(),
                eval_set=[(val_X, val_df["label"].tolist())],
                verbose=False,
            )
            # log XGBoost training loss curve to MLflow
            results = pipeline.named_steps["clf"].evals_result()
            for step, loss in enumerate(results["validation_0"]["mlogloss"]):
                mlflow.log_metric("xgb_val_mlogloss", loss, step=step)
        else:
            pipeline.fit(train_texts, train_df["label"].tolist())

        # ── Validate ─────────────────────────────────────────────────────────
        val_preds = pipeline.predict(val_texts)
        val_acc   = accuracy_score(val_df["label"], val_preds)
        val_f1    = f1_score(val_df["label"], val_preds, average="weighted")
        mlflow.log_metrics({"val_accuracy": val_acc, "val_f1_weighted": val_f1})
        logger.info(f"Val → accuracy: {val_acc:.4f}  F1: {val_f1:.4f}")

        # ── Test ─────────────────────────────────────────────────────────────
        test_preds    = pipeline.predict(test_texts)
        test_acc      = accuracy_score(test_df["label"], test_preds)
        test_f1       = f1_score(test_df["label"], test_preds, average="weighted")
        test_f1_macro = f1_score(test_df["label"], test_preds, average="macro")
        mlflow.log_metrics({
            "test_accuracy":    test_acc,
            "test_f1_weighted": test_f1,
            "test_f1_macro":    test_f1_macro,
        })
        logger.info(f"Test → accuracy: {test_acc:.4f}  F1: {test_f1:.4f}")

        # ── XGBoost feature importance plot ──────────────────────────────────
        if model_type == "xgboost":                       # ← NEW BLOCK
            tfidf        = pipeline.named_steps["tfidf"]
            xgb          = pipeline.named_steps["clf"]
            importances  = xgb.feature_importances_
            vocab        = tfidf.get_feature_names_out()
            top_idx      = importances.argsort()[-20:][::-1]
            top_words    = [vocab[i] for i in top_idx]
            top_scores   = importances[top_idx]

            fig2, ax = plt.subplots(figsize=(7, 5))
            ax.barh(top_words[::-1], top_scores[::-1], color="#7F77DD")
            ax.set_xlabel("Feature importance")
            ax.set_title("Top 20 TF-IDF features — XGBoost")
            plt.tight_layout()
            mlflow.log_figure(fig2, "feature_importance.png")
            plt.close(fig2)

        # ── Confusion matrix ─────────────────────────────────────────────────
        fig = plot_confusion_matrix(
            test_df["label"], test_preds, f"TF-IDF + {model_type.upper()}"
        )
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)

        # ── Classification report ─────────────────────────────────────────────
        report = classification_report(
            test_df["label"], test_preds, target_names=LABEL_NAMES
        )
        mlflow.log_text(report, "classification_report.txt")

        # ── Log model ─────────────────────────────────────────────────────────
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
            registered_model_name=f"thai-sentiment-tfidf-{model_type}",
        )

        run_id = mlflow.active_run().info.run_id
        logger.info(f"MLflow run ID: {run_id}")

    return pipeline, {"test_accuracy": test_acc, "test_f1_weighted": test_f1}

## 6. Load Processed Data
Read the cleaned train/val/test CSVs produced by the preprocessing notebook.

In [11]:
train = pd.read_csv("data/processed/train.csv")
val = pd.read_csv("data/processed/val.csv")
test = pd.read_csv("data/processed/test.csv")

logger.info(f"Train samples: {len(train)}")
logger.info(f"Validation samples: {len(val)}")
logger.info(f"Test samples: {len(test)}")

### 6.1 Inspect Data

In [12]:
train.head()

,text_clean,label,label_name
0,ไปจองมาแล้วนาจา Mitsubishi Attrage ได้หลังสงกร...,1,neu
1,เปิดศักราชใหม่! นายกฯ แถลงข่าวก่อนการแข่งขันศึ...,1,neu
2,บัตรสมาชิกลดได้อีกไหมคับ,1,neu
3,สนใจ new mazda2ครับ,1,neu
4,😍😍,0,neg


## 7. Train & Evaluate Models
Run the pipeline for each model type.

In [ ]:
train_and_evaluate(train, val, test, model_type="svm")

In [ ]:
train_and_evaluate(train, val, test, model_type="logreg")

In [ ]:
train_and_evaluate(train, val, test, model_type="xgboost")